# 7B 线步骤①云端冒烟（Kaggle T4）

链路：装环境 → 核验 CUDA → clone GitHub 仓库 → ModelScope 拉 AWQ 权重 → 跑 `eval/smoke_7b_load.py`。
核心产出：**steady tok/s**（决定 440 轮闭环真实成本）。所有输出写入 `/kaggle/working`，跑完用 `kaggle kernels output` 拉回。

注意：本 kernel 为 API 推送（batch 模式），单次上限 9 小时。

In [ ]:
# Cell 1: 环境安装（顺序不可反：先 torch 再 requirements，mistake #16）
!pip install -q "torch>=2.13,<2.16" --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
# Cell 2: 拉代码（本项目本体不在 kernel 源里，靠 GitHub；kaggle/ 只是推送壳）
!git clone https://github.com/Huzhenghua/emotion_model.git
%cd emotion_model
!git log --oneline -1

In [ ]:
# Cell 3: AWQ 权重下载（~5GB，ModelScope 直连快）
!modelscope download --model Qwen/Qwen2.5-7B-Instruct-AWQ --local_dir Qwen2.5-7B-Instruct-AWQ
!ls -la Qwen2.5-7B-Instruct-AWQ/

In [ ]:
# Cell 4: 步骤①冒烟——T4 上预期 steady 5-15 tok/s（exllama kernel；~2 tok/s = kernel 未挂上，回退纯 torch 路径）
!python eval/smoke_7b_load.py --model Qwen2.5-7B-Instruct-AWQ --max-new-tokens 128 2>&1 | tee /kaggle/working/smoke_7b.log

In [ ]:
# Cell 5: 产出归集（kaggle kernels output 拉回 /kaggle/working 的内容）
!ls -la /kaggle/working/